In [2]:
import torch
import pandas as pd
import sys
import os

current_dir = os.getcwd()
venv_path = os.path.abspath(os.path.join(current_dir, ".."))
src_path = os.path.join(venv_path, "Scripts", "src")
data_path = os.path.join(venv_path, "Scripts", "data")

sys.path.insert(0, src_path)

from dl_models import *

In [3]:
df = pd.read_csv(os.path.join(data_path, "processed", "labeled_data.csv"))
df = df.dropna(subset=["clean_text"])

texts = df["clean_text"].tolist()
labels = df["severity_id"].tolist()

In [4]:
vocab = build_vocab(texts)
vocab_size = len(vocab)
print("Vocab size:", vocab_size)

Vocab size: 20000


In [5]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

train_dataset = TextDataset(train_texts, train_labels, vocab)
test_dataset = TextDataset(test_texts, test_labels, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [6]:
print(set(labels))
print("Number of classes:", len(set(labels)))

{0, 1, 2, 3}
Number of classes: 4


In [7]:
import importlib
import dl_models
importlib.reload(dl_models)

<module 'dl_models' from 'd:\\AI-Based Airline Complaint Severity Detection & Service Risk Classification System\\venv\\Scripts\\src\\dl_models.py'>

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = len(set(labels))
model = LSTMModel(vocab_size, output_dim=num_classes).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    loss = train_model(model, train_loader, criterion, optimizer, device)
    acc = evaluate_model(model, test_loader, device)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}, Accuracy: {acc:.4f}")

Epoch 1, Loss: 0.8169, Accuracy: 0.7323
Epoch 2, Loss: 0.7177, Accuracy: 0.7290
Epoch 3, Loss: 0.5967, Accuracy: 0.7853
Epoch 4, Loss: 0.5028, Accuracy: 0.7942
Epoch 5, Loss: 0.4413, Accuracy: 0.7957


In [10]:
torch.save(model.state_dict(), "lstm_model.pt")